# 4.2.3 Perbandingan Pemeringkatan Pengeluaran Makanan, Nonmakanan, dan Total

Notebook ini menghasilkan dua kelompok analisis:

1. **Kesesuaian pemeringkatan aktual vs prediksi** untuk pengeluaran makanan dan nonmakanan.
2. **Kesesuaian pemeringkatan komponen aktual vs pengeluaran total aktual**.

Ukuran yang digunakan adalah **Kendall's Tau-b berbasis desil**. Perhitungan dilakukan pada masing-masing kabupaten/kota, kemudian diringkas dengan rata-rata antar kabupaten/kota pada masing-masing provinsi.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

BASE_PATH = Path.cwd()
OUTPUT_PATH = BASE_PATH / "output"

# File utama hasil prediksi komponen 2025.
# Notebook akan mencoba beberapa nama file agar tetap fleksibel.
candidate_files = [
    OUTPUT_PATH / "prediksi_komponen_desil_2025.xlsx",
    OUTPUT_PATH / "desil_prediksi_aktual_per_ruta.xlsx",
    BASE_PATH / "desil_prediksi_aktual_per_ruta (1).xlsx",
    BASE_PATH / "desil_prediksi_aktual_per_ruta TUNED.xlsx",
]

FILE_PRED = next((p for p in candidate_files if p.exists()), None)

if FILE_PRED is None:
    raise FileNotFoundError(
        "File prediksi komponen tidak ditemukan. "
        "Pastikan salah satu file prediksi komponen 2025 tersedia di folder output."
    )

print("File yang digunakan:", FILE_PRED)


In [ ]:
df = pd.read_excel(FILE_PRED)

print("Ukuran data:", df.shape)
print("\nKolom:")
print(df.columns.tolist())

display(df.head())


## Pemeriksaan dan standardisasi nama kolom

Notebook mengharapkan informasi berikut:

- provinsi
- kabupaten/kota
- model
- komponen (`food` / `nonfood`)
- desil komponen aktual
- desil komponen prediksi
- desil total aktual

Nama kolom akan dicari secara otomatis dari beberapa kemungkinan nama yang pernah digunakan pada output pemodelan.


In [ ]:
def find_col(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

COL_PROV = find_col(df.columns, ["province", "provinsi"])
COL_DIST = find_col(df.columns, ["district", "kabupaten/kota", "kabupaten_kota", "kabupaten"])
COL_MODEL = find_col(df.columns, ["model"])
COL_COMPONENT = find_col(df.columns, ["component", "komponen"])

COL_DEC_TRUE = find_col(df.columns, [
    "decile_component_true",
    "desil_component_true",
    "desil_komponen_aktual",
    "decile_component_actual"
])

COL_DEC_PRED = find_col(df.columns, [
    "decile_component_pred",
    "desil_component_pred",
    "desil_komponen_prediksi",
    "decile_component_prediction"
])

COL_TOTAL_TRUE = find_col(df.columns, [
    "decile_total_true",
    "desil_total_true",
    "desil_total_aktual",
    "decile_total_actual"
])

required = {
    "province": COL_PROV,
    "district": COL_DIST,
    "model": COL_MODEL,
    "component": COL_COMPONENT,
    "decile_component_true": COL_DEC_TRUE,
    "decile_component_pred": COL_DEC_PRED,
    "decile_total_true": COL_TOTAL_TRUE,
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Kolom berikut tidak ditemukan: {missing}\n"
        f"Kolom tersedia: {df.columns.tolist()}"
    )

print("Kolom yang digunakan:")
for k, v in required.items():
    print(f"{k:25s} -> {v}")


In [ ]:
# Standardisasi ke nama yang lebih sederhana
data = df.rename(columns={
    COL_PROV: "Provinsi",
    COL_DIST: "Kabupaten/Kota",
    COL_MODEL: "Model",
    COL_COMPONENT: "Komponen",
    COL_DEC_TRUE: "Desil_Aktual",
    COL_DEC_PRED: "Desil_Prediksi",
    COL_TOTAL_TRUE: "Desil_Total_Aktual",
}).copy()

# Rapikan label provinsi
prov_map = {
    "jogja": "D.I. Yogyakarta",
    "diy": "D.I. Yogyakarta",
    "D.I. Yogyakarta": "D.I. Yogyakarta",
    "jateng": "Jawa Tengah",
    "Jawa Tengah": "Jawa Tengah",
}

data["Provinsi"] = data["Provinsi"].replace(prov_map)

# Rapikan label komponen
data["Komponen"] = (
    data["Komponen"]
    .astype(str)
    .str.lower()
    .replace({
        "makanan": "food",
        "nonmakanan": "nonfood",
        "non-makanan": "nonfood",
    })
)

display(data.head())


## A. Kesesuaian desil aktual dan prediksi pengeluaran makanan/nonmakanan

Kendall's Tau-b dihitung pada setiap kombinasi **provinsi × kabupaten/kota × model × komponen**.

Semakin besar Tau-b, semakin sesuai pemeringkatan desil hasil prediksi dengan desil aktual.


In [ ]:
def safe_kendall(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 2:
        return np.nan
    return kendalltau(tmp["x"], tmp["y"]).correlation

tau_pred_kab = (
    data
    .groupby(
        ["Provinsi", "Kabupaten/Kota", "Model", "Komponen"],
        as_index=False
    )
    .apply(
        lambda g: pd.Series({
            "Tau_b": safe_kendall(
                g["Desil_Aktual"],
                g["Desil_Prediksi"]
            ),
            "n": len(g)
        }),
        include_groups=False
    )
    .reset_index(drop=True)
)

display(tau_pred_kab.head(10))


In [ ]:
# Lihat model yang tersedia
print("Model yang tersedia:")
for m in sorted(tau_pred_kab["Model"].dropna().astype(str).unique()):
    print("-", m)


### Pilih model utama

Secara konseptual, untuk bagian ini **Total XGB tidak dimasukkan** karena tidak menghasilkan prediksi makanan dan nonmakanan.

Jika nama model pada file berbeda, sesuaikan daftar `model_utama_komponen` setelah melihat output cell sebelumnya.


In [ ]:
# Nama model final yang ingin ditampilkan.
# Notebook juga mempertahankan hanya nama yang memang tersedia dalam file.

model_utama_komponen = [
    "XGB_Components",
    "MultiOutputTree_XGB",
    "RF_Components",
    "Multivariate_RF",
]

tersedia = set(tau_pred_kab["Model"].astype(str).unique())
model_dipakai = [m for m in model_utama_komponen if m in tersedia]

# Fallback: jika nama final tidak ditemukan, pakai seluruh model komponen
# kecuali model total.
if len(model_dipakai) == 0:
    model_dipakai = [
        m for m in sorted(tersedia)
        if "total" not in m.lower()
    ]

print("Model yang dipakai:", model_dipakai)

tau_pred_kab_utama = tau_pred_kab[
    tau_pred_kab["Model"].isin(model_dipakai)
].copy()


In [ ]:
# Tabel per kabupaten/kota:
# satu baris = provinsi × kabupaten × model
tabel_tau_pred_kab = (
    tau_pred_kab_utama
    .pivot_table(
        index=["Provinsi", "Kabupaten/Kota", "Model"],
        columns="Komponen",
        values="Tau_b"
    )
    .reset_index()
    .rename(columns={
        "food": "Tau-b Makanan",
        "nonfood": "Tau-b Nonmakanan"
    })
)

display(tabel_tau_pred_kab.round(4))


In [ ]:
# Ringkasan rata-rata antar kabupaten/kota per provinsi dan model
ringkasan_tau_pred = (
    tabel_tau_pred_kab
    .groupby(["Provinsi", "Model"], as_index=False)[
        ["Tau-b Makanan", "Tau-b Nonmakanan"]
    ]
    .mean()
)

# Tambahkan komponen dengan kesesuaian prediksi lebih tinggi
ringkasan_tau_pred["Komponen Lebih Tinggi"] = np.where(
    ringkasan_tau_pred["Tau-b Makanan"] > ringkasan_tau_pred["Tau-b Nonmakanan"],
    "Makanan",
    np.where(
        ringkasan_tau_pred["Tau-b Nonmakanan"] > ringkasan_tau_pred["Tau-b Makanan"],
        "Nonmakanan",
        "Sama"
    )
)

display(ringkasan_tau_pred.round(4))


**Usulan nama tabel Bab IV:**

**Tabel 4.x Kesesuaian Pemeringkatan Pengeluaran Makanan dan Nonmakanan antara Nilai Aktual dan Prediksi di Provinsi D.I. Yogyakarta dan Jawa Tengah**


## B. Kesesuaian desil pengeluaran makanan/nonmakanan aktual terhadap desil pengeluaran total aktual

Bagian ini **bukan ukuran akurasi prediksi model**. Tujuannya melihat seberapa selaras pemeringkatan berdasarkan masing-masing komponen pengeluaran dengan pemeringkatan berdasarkan pengeluaran total.

Karena hubungan ini berasal dari nilai aktual, hasilnya tidak bergantung pada model.


In [ ]:
# Hitung pada setiap kabupaten/kota.
# Karena data aktual berulang untuk setiap model, cukup hitung hubungan aktual
# pada setiap model lalu ambil satu hasil untuk tiap kabupaten × komponen.
# Nilainya seharusnya sama antar model karena variabel yang dibandingkan aktual.

tau_total_tmp = (
    data
    .groupby(
        ["Provinsi", "Kabupaten/Kota", "Model", "Komponen"],
        as_index=False
    )
    .apply(
        lambda g: pd.Series({
            "Tau_b_Komponen_Total": safe_kendall(
                g["Desil_Aktual"],
                g["Desil_Total_Aktual"]
            )
        }),
        include_groups=False
    )
    .reset_index(drop=True)
)

# Ambil satu nilai per kabupaten × komponen
tau_total_kab = (
    tau_total_tmp
    .groupby(
        ["Provinsi", "Kabupaten/Kota", "Komponen"],
        as_index=False
    )["Tau_b_Komponen_Total"]
    .first()
)

tabel_tau_total_kab = (
    tau_total_kab
    .pivot_table(
        index=["Provinsi", "Kabupaten/Kota"],
        columns="Komponen",
        values="Tau_b_Komponen_Total"
    )
    .reset_index()
    .rename(columns={
        "food": "Tau-b Makanan vs Total",
        "nonfood": "Tau-b Nonmakanan vs Total"
    })
)

display(tabel_tau_total_kab.round(4))


In [ ]:
# Ringkasan rata-rata antar kabupaten/kota per provinsi
ringkasan_tau_total = (
    tabel_tau_total_kab
    .groupby("Provinsi", as_index=False)[
        ["Tau-b Makanan vs Total", "Tau-b Nonmakanan vs Total"]
    ]
    .mean()
)

ringkasan_tau_total["Komponen Lebih Selaras dengan Total"] = np.where(
    ringkasan_tau_total["Tau-b Makanan vs Total"] >
    ringkasan_tau_total["Tau-b Nonmakanan vs Total"],
    "Makanan",
    np.where(
        ringkasan_tau_total["Tau-b Nonmakanan vs Total"] >
        ringkasan_tau_total["Tau-b Makanan vs Total"],
        "Nonmakanan",
        "Sama"
    )
)

display(ringkasan_tau_total.round(4))


**Usulan nama tabel Bab IV:**

**Tabel 4.x Kesesuaian Pemeringkatan Pengeluaran Makanan dan Nonmakanan terhadap Pengeluaran Total di Provinsi D.I. Yogyakarta dan Jawa Tengah**


## C. Ringkasan tambahan per provinsi

Cell berikut menyusun ringkasan yang dapat digunakan sebagai bahan narasi:

- rata-rata Tau-b aktual vs prediksi makanan/nonmakanan;
- rata-rata Tau-b komponen aktual vs total aktual.


In [ ]:
display(
    ringkasan_tau_pred
    .sort_values(["Provinsi", "Model"])
    .round(4)
)

display(
    ringkasan_tau_total
    .sort_values("Provinsi")
    .round(4)
)


## D. Simpan seluruh hasil ke Excel


In [ ]:
FILE_OUTPUT = OUTPUT_PATH / "hasil_4_2_3_perbandingan_pemeringkatan.xlsx"

with pd.ExcelWriter(FILE_OUTPUT, engine="openpyxl") as writer:

    # Aktual vs prediksi komponen
    tabel_tau_pred_kab.round(4).to_excel(
        writer,
        sheet_name="Prediksi_per_kab",
        index=False
    )

    ringkasan_tau_pred.round(4).to_excel(
        writer,
        sheet_name="Prediksi_rata2_prov",
        index=False
    )

    # Komponen aktual vs total aktual
    tabel_tau_total_kab.round(4).to_excel(
        writer,
        sheet_name="Komponen_total_per_kab",
        index=False
    )

    ringkasan_tau_total.round(4).to_excel(
        writer,
        sheet_name="Komponen_total_prov",
        index=False
    )

print("Tersimpan:", FILE_OUTPUT)


## Cara membaca hasil untuk Bab IV

**Aktual vs prediksi komponen**

- Tau-b lebih besar menunjukkan desil prediksi lebih sesuai dengan desil aktual.
- Bandingkan makanan dan nonmakanan untuk melihat komponen mana yang pemeringkatannya lebih mudah direproduksi model.
- Bandingkan model *single-output* dan *multi-output* tanpa langsung menganggap nilai tertinggi sebagai bukti bahwa satu pendekatan selalu lebih baik.

**Komponen aktual vs total aktual**

- Tau-b makanan vs total menunjukkan seberapa selaras peringkat kesejahteraan berdasarkan pengeluaran makanan dengan total.
- Tau-b nonmakanan vs total menunjukkan hal yang sama untuk pengeluaran nonmakanan.
- Analisis ini bersifat deskriptif-struktural dan **bukan** ukuran ketepatan prediksi model.
